# BT Models Testing: AFL Data

In this notebook, we will be testing the various Bradley-Terry models in our package using data from the Australian Football League (AFL). The data is sourced from [a repository by Kareen (2025)](https://github.com/akareen/AFL-Data-Analysis). Testing is also done here for the various utility functions for conveniently formatting the AFL data for use in our package.

In [ ]:
import os
import sys

root_dir = os.path.dirname(os.path.abspath(os.curdir))
src_dir = os.path.join(root_dir, 'src', 'python')
sys.path.append(src_dir)
# print(sys.path[-1])

In [2]:
import numpy as np
from numpy import ndarray, exp, log
import pandas as pd
from pandas import DataFrame

from StaticModel import *
from DynamicModel import *
from utils import *

# Loading Data

For the sake of demonstration, we will use the data from the 2015-2016 seasons, so that there are two years of data for the discrete dyanmic models. The data is already separated into different CSVs by year.

In [3]:
from data_utils_afl import *

path = "../data/akareen-matches/matches_2015.csv"
afl_raw = pd.read_csv(path)
afl_raw.head()

,round_num,venue,date,year,attendance,team_1_team_name,team_1_q1_goals,team_1_q1_behinds,team_1_q2_goals,team_1_q2_behinds,...,team_1_final_behinds,team_2_team_name,team_2_q1_goals,team_2_q1_behinds,team_2_q2_goals,team_2_q2_behinds,team_2_q3_goals,team_2_q3_behinds,team_2_final_goals,team_2_final_behinds
0,1,M.C.G.,2015-04-02 19:20,2015,6:20 PM,Carlton,4,7,6,9,...,12,Richmond,2,3,7,10,9,13,15,15
1,1,M.C.G.,2015-04-04 13:40,2015,12:40 PM,Melbourne,2,5,7,7,...,13,Gold Coast,1,2,3,6,8,8,13,11
2,1,Stadium Australia,2015-04-04 16:35,2015,3:35 PM,Sydney,0,1,1,7,...,12,Essendon,4,3,5,5,9,6,9,6
3,1,Gabba,2015-04-04 18:20,2015,NaN,Brisbane Lions,2,3,3,5,...,8,Collingwood,3,3,7,6,12,11,12,14
4,1,Docklands,2015-04-04 19:20,2015,6:20 PM,Western Bulldogs,4,0,7,5,...,13,West Coast,3,1,7,1,12,2,14,3


In [4]:
path = "../data/akareen-matches/"
unique_teams = get_unique_teams(path, 2015, 2016)
# print(unique_teams)
print(len(unique_teams))
print(sorted(list(unique_teams)))

18
['Adelaide', 'Brisbane Lions', 'Carlton', 'Collingwood', 'Essendon', 'Fremantle', 'Geelong', 'Gold Coast', 'Greater Western Sydney', 'Hawthorn', 'Melbourne', 'North Melbourne', 'Port Adelaide', 'Richmond', 'St Kilda', 'Sydney', 'West Coast', 'Western Bulldogs']


In [5]:
path = "../data/akareen-matches/"
unique_venues = get_unique_venues(path, 2015, 2016)
# print(unique_venues)
print(len(unique_venues))
print(sorted(list(unique_venues)))

17
['Adelaide Oval', 'Bellerive Oval', 'Carrara', "Cazaly's Stadium", 'Docklands', 'Gabba', 'Kardinia Park', 'M.C.G.', 'Manuka Oval', 'Marrara Oval', 'S.C.G.', 'Stadium Australia', 'Subiaco', 'Sydney Showground', 'Traeger Park', 'Wellington', 'York Park']


In [6]:
path = "../data/akareen-matches/"
df = load_data_afl(path, start_year=2015, end_year=2016)
df

,home,away,home_score,away_score,is_neutral,year
0,Carlton,Richmond,78,105,0,2015
1,Melbourne,Gold Coast,115,89,0,2015
2,Sydney,Essendon,72,60,0,2015
3,Brisbane Lions,Collingwood,74,86,0,2015
4,Western Bulldogs,West Coast,97,87,0,2015
...,...,...,...,...,...,...
408,Hawthorn,Western Bulldogs,84,107,0,2016
409,Sydney,Adelaide,118,82,0,2016
410,Geelong,Sydney,60,97,0,2016
411,Greater Western Sydney,Western Bulldogs,83,89,0,2016


## Testing Dynamic Models on AFL data

In [7]:
df_van = df.drop("is_neutral", axis=1)
df_van.head()

,home,away,home_score,away_score,year
0,Carlton,Richmond,78,105,2015
1,Melbourne,Gold Coast,115,89,2015
2,Sydney,Essendon,72,60,2015
3,Brisbane Lions,Collingwood,74,86,2015
4,Western Bulldogs,West Coast,97,87,2015


In [8]:
df_time = make_time_blocked_win_matrix(df_van)
df_time[2015].head()

,Adelaide,Brisbane Lions,Carlton,Collingwood,Essendon,Fremantle,Geelong,Gold Coast,Greater Western Sydney,Hawthorn,Melbourne,North Melbourne,Port Adelaide,Richmond,St Kilda,Sydney,West Coast,Western Bulldogs
Adelaide,0.0,2.0,1.0,1.0,1.0,0.0,0.0,2.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0
Brisbane Lions,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
Carlton,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
Collingwood,0.0,1.0,2.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
Essendon,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [9]:
dyvan = DyVANBT()
dyvan.fit(df_time, verbose=True)

Optimization terminated successfully.
         Current function value: 203.059835
         Iterations: 21
         Function evaluations: 35
         Gradient evaluations: 35
Successfully fit. Summary below:

  message: Optimization terminated successfully.
  success: True
   status: 0
      fun: 203.05983483531583
        x: [ 2.428e+00  0.000e+00 ...  3.505e+00  3.879e+00]
      nit: 21
      jac: [-4.129e-07 -1.707e-06 ... -3.876e-06  2.877e-06]
 hess_inv: [[ 9.196e-01  2.325e-02 ... -1.937e-02  6.102e-02]
            [ 2.325e-02  8.661e-01 ...  5.458e-04 -2.372e-02]
            ...
            [-1.937e-02  5.458e-04 ...  7.606e-01  6.865e-03]
            [ 6.102e-02 -2.372e-02 ...  6.865e-03  7.860e-01]]
     nfev: 35
     njev: 35


DyVANBT()

In [10]:
check_model_grad(dyvan)

np.float64(4.8389114181912935e-05)

In [11]:
dyvan.get_ranking()

,Team,2015,2016,Average
0,Fremantle,3.310136,0.572143,1.941139
1,West Coast,3.223050,3.505183,3.364117
2,Hawthorn,3.187919,3.747167,3.467543
3,Sydney,2.786247,3.938574,3.362410
4,Richmond,2.595167,1.629190,2.112178
5,North Melbourne,2.477352,2.869379,2.673366
6,Adelaide,2.427883,3.674126,3.051005
7,Western Bulldogs,2.337806,3.878930,3.108368
8,Port Adelaide,2.309597,2.121839,2.215718
9,Geelong,2.215571,3.984954,3.100262


## Compare to individual static BT models

In [12]:
data_2015 = df_time[2015]
# print(data_2015.head())
mod_2015 = VANBT()
mod_2015.fit(data_2015)
mod_2015.get_ranking()

,Teams,Ability
0,Fremantle,3.310135
1,West Coast,3.223051
2,Hawthorn,3.187918
3,Sydney,2.786247
4,Richmond,2.595167
5,North Melbourne,2.477352
6,Adelaide,2.427883
7,Western Bulldogs,2.337805
8,Port Adelaide,2.309597
9,Geelong,2.215570


In [14]:
dyvan.get_ranking(2016)

,Team,2015,2016,Average
0,Geelong,2.215571,3.984954,3.100262
1,Sydney,2.786247,3.938574,3.362410
2,Western Bulldogs,2.337806,3.878930,3.108368
3,Hawthorn,3.187919,3.747167,3.467543
4,Greater Western Sydney,1.649512,3.696901,2.673207
5,Adelaide,2.427883,3.674126,3.051005
6,West Coast,3.223050,3.505183,3.364117
7,North Melbourne,2.477352,2.869379,2.673366
8,St Kilda,0.799539,2.563768,1.681653
9,Port Adelaide,2.309597,2.121839,2.215718


In [13]:
data_2016 = df_time[2016]
# print(data_2015.head())
mod_2016 = VANBT()
mod_2016.fit(data_2016)
mod_2016.get_ranking()

,Teams,Ability
0,Geelong,3.984954
1,Sydney,3.938575
2,Western Bulldogs,3.878929
3,Hawthorn,3.747168
4,Greater Western Sydney,3.696902
5,Adelaide,3.674125
6,West Coast,3.505184
7,North Melbourne,2.869379
8,St Kilda,2.563766
9,Port Adelaide,2.121838
